### 機械学習特論 第5回 分類 k近傍法 アヤメのデータ、mnistデータ、fashion-mnistデータの分類

In [ ]:
# google colab で実行する場合は、次の行の先頭の # を削除してこのブロックを実行する
#!pip install japanize-matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.svm import SVC
import statistics
import japanize_matplotlib

### アヤメのデータを k近傍法で学習して評価

In [ ]:
# sklearnデータセットに収録されたiris(アヤメ)のデータセットをロード
from sklearn.datasets import load_iris
iris_data = load_iris()

# 学習用データのデータフレームを作成 (targetが1か2のデータである先頭100件を抽出)
x = pd.DataFrame(iris_data["data"],columns=iris_data["feature_names"])[50:150]
y = pd.DataFrame(iris_data["target"],columns=["target"])[50:150] # species 1 2 を抽出

# データフレームの情報を表示
print('feature shape:', x.shape) # 説明変数の形状
print('feature names:', iris_data["feature_names"]) # 説明変数の列名
print('target shape:', y.shape, y['target'].unique()) # 正解ラベルの形状とクラス名一覧

In [ ]:
# データを学習用と検証用に分割
x_train, x_test, y_train, y_test = \
    train_test_split(x, y, test_size=0.25, random_state=1, stratify=y) # 検証用データに25%を割当て、各画像を均等に分割 (straify=y)
print(len(x_train), len(x_test), len(y_train), len(y_test)) # それぞれのデータ数を確認

# ペアプロットを表示
_plot = x_train.copy()  # 学習データをペアプロット用にコピー
_plot['species'] = y_train  # コピーした学習データに正解ラベルの列を追加
sns.pairplot(_plot, hue='species', # 品種(species)で色分け
                    palette={1: 'red', 2: 'blue'},   # 各クラスの色を指定
                    height=1.5)  # ペアプロットの各グラフの高さを指定 (全体ではない)
plt.show()

In [ ]:
# k近傍法のモデルを学習データを利用して学習
clf_3 = KNeighborsClassifier(n_neighbors=3, # k を指定 (デフォルトは 5)
                             weights='uniform',  # 距離を考慮しない(uniform:デフォルト)、する(distance)
                             algorithm='auto', # 近傍点計算アルゴリズム (auto:デフォルト,ball_tree,kd_tree,brute)
                             leaf_size=30,  # ball_tree,kd_tree指定時のリーフサイズの設定 (デフォルトは 30)
                             p=2)  # 距離計算の次元 (2:デフォルト、1)
clf_3 = clf_3.fit(x_train, y_train['target'])  # * y_trainの列指定はwarning回避のため *

# 学習したモデルの性能(正答率)を学習用データで評価
predict_3_train = clf_3.predict(x_train)
print('knn (k=3, train data) : accuracy_score', metrics.accuracy_score(y_train, predict_3_train))

# 学習したモデルの性能(正答率)を検証用データで評価
predict_3_test = clf_3.predict(x_test)
print('knn (k=3, test data) : accuracy_score', metrics.accuracy_score(y_test, predict_3_test))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
i = 0
k = []
test_accuracy = []
train_accuracy = []
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=1)
print(len(x_train), len(x_test), len(y_train), len(y_test))
print(x_train.shape, y_train.shape)

for _k in range(1, 16):
    # k近傍法で学習データを学習 (kは1-15まで設定)
    clf = KNeighborsClassifier(n_neighbors=_k, weights='uniform', algorithm='auto', p=2)
    clf = clf.fit(x_train, y_train['target'])  # * y_trainの列指定はwarning回避のため *

    # 学習用データに対する正解率を計算
    predict_train = clf.predict(x_train)
    _train_accuracy = metrics.accuracy_score(y_train, predict_train)
    
    # 検証用データに対する正解率を計算
    predict_test = clf.predict(x_test)
    _test_accuracy = metrics.accuracy_score(y_test, predict_test)

    # 学習用データと検証用データにおける正解率を表示
    print(f'k={_k}', '(train) 正解率', _train_accuracy, '(test) 正解率', _test_accuracy)

    # kを変えた場合の学習用データと検証用データの正解率グラフのデータを準備
    k.append(_k)
    train_accuracy.append(_train_accuracy)
    test_accuracy.append(_test_accuracy)


# 各決定係数をヒストグラムに表示 (ビンの幅・数を揃えて、透過色で重ね合わせて表示)
ax.set_title(f'k近傍法の k と正解率')
ax.plot(k, test_accuracy, label='test_accuracy')
ax.plot(k, train_accuracy, label='train_accuracy')
ax.set_xticks(np.arange(1, 16, step=1))
ax.legend()  # loc='upper left'

plt.show()

### mnistデータを k近傍法で学習して評価

In [ ]:
# 手書き文字のデータセットをダウンロードして、実験用データを準備
from sklearn.datasets import fetch_openml
mnist_data = fetch_openml('mnist_784', version=1)
_x = np.array(mnist_data['data'].astype(np.float32))
_y = np.array(mnist_data['target'].astype(np.int32))
# 処理時間の都合により、70000枚のうち7000枚を利用
_, x, _, y = train_test_split(_x, _y, test_size=0.1, random_state=1)

In [ ]:
# ダウンロードしたデータセットを確認
print('xの形状:', x.shape)
print('yの形状:', y.shape, type(y))
print(f'0番目のデータの形状: x[0].shape={x[0].shape}, ターゲット: y[0]={y[0]}')
for i in range(28):
    for j in range(28):
        print(f'{int(x[0][i*28+j]):03}', end=' ')
    print()

# データセットに含まれる文字のイメージを表示 (5x2のレイアウトで最初の10画像)
rows, cols = 2, 5
fig, ax = plt.subplots(rows, cols, figsize=(10.0, 4.0), tight_layout=True)
for i in range(rows * cols):
    ax[i // cols][i % cols].imshow(x[i].reshape(28,28), cmap=plt.cm.gray_r)
    ax[i // cols][i % cols].set_title(f'target: {y[i]}')
plt.show()

In [ ]:
# データセットに含まれる各数字の画像数を確認
for i in range(10):
    n = 0
    for j in range(len(y)):
        if i == y[j]:
            n = n + 1
    print(f'{i}: {n}, ', end='')  # 各数字の画像数にバラツキあり

# データを学習用と検証用に分割
x_train, x_test, y_train, y_test = \
    train_test_split(x, y, test_size=0.25, random_state=1, stratify=y) # 検証用データに25%を割当て、各画像を均等に分割 (straify=y)
print('\n', len(x_train), len(x_test), len(y_train), len(y_test)) # それぞれのデータ数を確認

In [ ]:
# k近傍法のモデルを学習データを利用して学習
clf_3 = KNeighborsClassifier(n_neighbors=3, # k を指定 (デフォルトは 5)
                             weights='uniform',  # 距離を考慮しない(uniform:デフォルト)、する(distance)
                             algorithm='auto', # 近傍点計算アルゴリズム (auto:デフォルト,ball_tree,kd_tree,brute)
                             leaf_size=30,  # ball_tree,kd_tree指定時のリーフサイズの設定 (デフォルトは 30)
                             p=2)  # 距離計算の次元 (2:デフォルト、1)
clf_3 = clf_3.fit(x_train, y_train)  # * y_trainの列指定はwarning回避のため *

# 学習したモデルの性能(正答率)を学習用データで評価
predict_3_train = clf_3.predict(x_train)
print('knn (k=3, train data) : accuracy_score', metrics.accuracy_score(y_train, predict_3_train))

# 学習したモデルの性能(正答率)を検証用データで評価
predict_3_test = clf_3.predict(x_test)
print('knn (k=3, test data) : accuracy_score', metrics.accuracy_score(y_test, predict_3_test))

In [ ]:
# 各数字の予測精度を確認
_p = predict_3_test.tolist()
for j in range(10):
    n = 0
    c = 0
    for i in range(len(_p)):
        if j == y_test[i]:
            n = n + 1
            if j == _p[i]:
                c = c + 1
    print(f'{j} : total {n} correct {c} precistion {c/n}')

# 混同行列を作成して、正解を横軸、予測を縦軸としてヒートマップを作成
from sklearn.metrics import confusion_matrix
conf_mat = confusion_matrix(y_test, _p) 
plt.figure(figsize = (12, 10))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap="Reds")
plt.ylabel('correct label')
plt.xlabel('predict label')
plt.show()

In [ ]:
# 予測誤った画像のインデックスと正解と誤答を確認
for i in range(len(_p)):
    if _p[i] != y_test[i]:
        print(i, 'gt', y_test[i], '-> predict', _p[i])

# データセットに含まれる予測を誤った文字のイメージを表示 (5x2のレイアウトで)
img_no = [6, 7, 15, 41]
rows, cols = 1, 4
fig, ax = plt.subplots(rows, cols, figsize=(12.0, 3.0), tight_layout=True)
for i in range(len(img_no)):
    ax[i].imshow(x_test[img_no[i]].reshape(28,28), cmap="binary", interpolation="nearest")
    ax[i].set_title(f'gt: {y_test[img_no[i]]} -> predict:{_p[img_no[i]]}')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
i = 0
k = []
test_accuracy = []
train_accuracy = []
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=1)
print(len(x_train), len(x_test), len(y_train), len(y_test))
print(x_train.shape, y_train.shape)

for _k in range(1, 16):
    # k近傍法で学習データを学習 (kは1-15まで設定)
    clf = KNeighborsClassifier(n_neighbors=_k, weights='uniform', algorithm='auto', p=2)
    clf = clf.fit(x_train, y_train)  # * y_trainの列指定はwarning回避のため *

    # 学習用データに対する正解率を計算
    predict_train = clf.predict(x_train)
    _train_accuracy = metrics.accuracy_score(y_train, predict_train)
    
    # 検証用データに対する正解率を計算
    predict_test = clf.predict(x_test)
    _test_accuracy = metrics.accuracy_score(y_test, predict_test)

    # 学習用データと検証用データにおける正解率を表示
    print(f'k={_k}', '(train) 正解率', _train_accuracy, '(test) 正解率', _test_accuracy)

    # kを変えた場合の学習用データと検証用データの正解率グラフのデータを準備
    k.append(_k)
    train_accuracy.append(_train_accuracy)
    test_accuracy.append(_test_accuracy)


# 各決定係数をヒストグラムに表示 (ビンの幅・数を揃えて、透過色で重ね合わせて表示)
ax.set_title(f'k近傍法の k と正解率')
ax.plot(k, test_accuracy, label='test_accuracy')
ax.plot(k, train_accuracy, label='train_accuracy')
ax.set_xticks(np.arange(1, 16, step=1))
ax.legend()  # loc='upper left'

plt.show()

### fashion-mnistデータを k近傍法で学習して評価

In [ ]:
# Fashion-MNISTデータセットをダウンロードして、実験用データを準備
from sklearn.datasets import fetch_openml
mnist_data = fetch_openml('Fashion-MNIST')
_x = np.array(mnist_data['data'].astype(np.float32))
_y = np.array(mnist_data['target'].astype(np.int32))
# 処理時間の都合から、70000枚の画像のうち7000枚を対象とする (各画像を均等に (straify=y))
_, x, _, y = train_test_split(_x, _y, test_size=0.1, random_state=1, stratify=_y) 

In [ ]:
# ダウンロードしたデータセットを確認
print('xの形状:', x.shape)
print('yの形状:', y.shape, type(y))

# データセットに含まれる文字のイメージを表示 (2x5のレイアウトで最初の10画像)
rows, cols = 5, 2
fig, ax = plt.subplots(rows, cols, figsize=(4.0, 10.0), tight_layout=True)
for i in range(rows * cols):
    ax[i // cols][i % cols].imshow(x[i].reshape(28,28), cmap=plt.cm.gray_r)
    ax[i // cols][i % cols].set_title(f'target: {y[i]}')
plt.show()

In [ ]:
# データセットに含まれる各画像数を確認
for i in range(10):
    n = 0
    for j in range(len(y)):
        if i == y[j]:
            n = n + 1
    print(f'{i}: {n}, ', end='')  # 各数字の画像数にバラツキあり

# データを学習用と検証用に分割 (検証用データに25%を割当て)
x_train, x_test, y_train, y_test = \
    train_test_split(x, y, test_size=0.25, random_state=1, stratify=y) # 検証用データに25%を割当て、各画像を均等に分割 (straify=y)
print('\n', len(x_train), len(x_test), len(y_train), len(y_test)) # それぞれのデータ数を確認

In [ ]:
# k近傍法のモデルを学習データを利用して学習
clf_3 = KNeighborsClassifier(n_neighbors=3, # k を指定 (デフォルトは 5)
                             weights='uniform',  # 距離を考慮しない(uniform:デフォルト)、する(distance)
                             algorithm='auto', # 近傍点計算アルゴリズム (auto:デフォルト,ball_tree,kd_tree,brute)
                             leaf_size=30,  # ball_tree,kd_tree指定時のリーフサイズの設定 (デフォルトは 30)
                             p=2)  # 距離計算の次元 (2:デフォルト、1)
clf_3 = clf_3.fit(x_train, y_train)  # * y_trainの列指定はwarning回避のため *

# 学習したモデルの性能(正答率)を学習用データで評価
predict_3_train = clf_3.predict(x_train)
print('knn (k=3, train data) : accuracy_score', metrics.accuracy_score(y_train, predict_3_train))

# 学習したモデルの性能(正答率)を検証用データで評価
predict_3_test = clf_3.predict(x_test)
print('knn (k=3, test data) : accuracy_score', metrics.accuracy_score(y_test, predict_3_test))

In [ ]:
# 各画像の予測精度を確認
_p = predict_3_test.tolist()
for j in range(10):
    n = 0
    c = 0
    for i in range(len(_p)):
        if j == y_test[i]:
            n = n + 1
            if j == _p[i]:
                c = c + 1
    print(f'{j} : total {n} correct {c} precistion {c/n}')

# 混同行列を作成して、正解を縦軸、予測を横軸としてヒートマップを作成
from sklearn.metrics import confusion_matrix
conf_mat = confusion_matrix(y_test, _p) 
plt.figure(figsize = (12, 10))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap="Reds")
plt.ylabel('correct label')
plt.xlabel('predict label')
plt.show()

In [ ]:
# 予測誤った画像のインデックスと正解と誤答を確認
for i in range(len(_p)):
    if _p[i] != y_test[i]:
        print(i, 'gt', y_test[i], '-> predict', _p[i])

# データセットに含まれる予測を誤った文字のイメージを表示 (5x2のレイアウトで)
img_no = [53, 1667, 13, 130, 2772, 834, 321, 365, 455, 389]
img_p = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rows, cols = 1, 10
fig, ax = plt.subplots(rows, cols, figsize=(15.0, 2.0), tight_layout=True)
for i in range(len(img_no)):
    ax[i].imshow(x[img_no[i]].reshape(28,28), cmap="binary", interpolation="nearest")
    ax[i].tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)
    ax[i].set_xlabel(f'{y[img_no[i]]} -> {img_p[i]}')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
i = 0
k = []
test_accuracy = []
train_accuracy = []
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=1)
print(len(x_train), len(x_test), len(y_train), len(y_test))
print(x_train.shape, y_train.shape)

for _k in range(1, 16):
    # k近傍法で学習データを学習 (kは1-15まで設定)
    clf = KNeighborsClassifier(n_neighbors=_k, weights='uniform', algorithm='auto', p=2)
    clf = clf.fit(x_train, y_train)  # * y_trainの列指定はwarning回避のため *

    # 学習用データに対する正解率を計算
    predict_train = clf.predict(x_train)
    _train_accuracy = metrics.accuracy_score(y_train, predict_train)
    
    # 検証用データに対する正解率を計算
    predict_test = clf.predict(x_test)
    _test_accuracy = metrics.accuracy_score(y_test, predict_test)

    # 学習用データと検証用データにおける正解率を表示
    print(f'k={_k}', '(train) 正解率', _train_accuracy, '(test) 正解率', _test_accuracy)

    # kを変えた場合の学習用データと検証用データの正解率グラフのデータを準備
    k.append(_k)
    train_accuracy.append(_train_accuracy)
    test_accuracy.append(_test_accuracy)


# 各決定係数をヒストグラムに表示 (ビンの幅・数を揃えて、透過色で重ね合わせて表示)
ax.set_title(f'k近傍法の k と正解率')
ax.plot(k, test_accuracy, label='test_accuracy')
ax.plot(k, train_accuracy, label='train_accuracy')
ax.set_xticks(np.arange(1, 16, step=1))
ax.legend()  # loc='upper left'

plt.show()